# 02 · Data distribution

Visual EDA over the seeded `transactions` table. Aggregations run in MySQL so we do not pull 7.3M rows into memory.

Focus: volume over time, corridors, amounts, and per-user behaviour.

In [ ]:
from cross_model_drift.notebook import setup_eda

nb = setup_eda()
config, table, read_sql, show = nb.config, nb.table, nb.read_sql, nb.show
config

## Snapshot

In [ ]:
overview = read_sql(
    f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT sender_id) AS n_senders,
        COUNT(DISTINCT payout_country) AS n_countries,
        COUNT(DISTINCT payout_currency) AS n_currencies,
        COUNT(DISTINCT CONCAT(payout_country, '-', payout_currency)) AS n_corridors,
        MIN(created) AS created_min,
        MAX(created) AS created_max,
        MIN(amount_usd) AS amount_min,
        MAX(amount_usd) AS amount_max,
        AVG(amount_usd) AS amount_avg,
        MIN(fee_usd) AS fee_min,
        MAX(fee_usd) AS fee_max,
        AVG(fee_usd) AS fee_avg,
        AVG(fee_usd / NULLIF(amount_usd, 0)) AS fee_ratio_avg,
        SUM(anti_fraud_status = 'positive') / COUNT(*) AS fraud_rate
    FROM `{table}`
    """
)
overview.T.rename(columns={0: "value"})

## Volume over time

In [ ]:
monthly = read_sql(
    f"""
    SELECT
        DATE_FORMAT(created, '%Y-%m-01') AS month,
        COUNT(*) AS n_tx,
        SUM(amount_usd) AS volume_usd,
        SUM(anti_fraud_status = 'positive') / COUNT(*) AS fraud_rate
    FROM `{table}`
    GROUP BY DATE_FORMAT(created, '%Y-%m-01')
    ORDER BY month
    """
)
monthly["month"] = pd.to_datetime(monthly["month"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
sns.barplot(data=monthly, x="month", y="n_tx", ax=axes[0], color="#4C78A8")
axes[0].set_title("Transactions per month")
axes[0].set_xlabel("")
axes[0].set_ylabel("transactions")
axes[0].tick_params(axis="x", rotation=30)
axes[0].yaxis.set_major_formatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}k")

sns.barplot(data=monthly, x="month", y="volume_usd", ax=axes[1], color="#F58518")
axes[1].set_title("USD volume per month")
axes[1].set_xlabel("")
axes[1].set_ylabel("USD")
axes[1].tick_params(axis="x", rotation=30)
axes[1].yaxis.set_major_formatter(lambda x, _: f"${x/1e6:.0f}M")
show(fig)
monthly

In [ ]:
weekly = read_sql(
    f"""
    SELECT
        DATE(DATE_SUB(created, INTERVAL WEEKDAY(created) DAY)) AS week_start,
        COUNT(*) AS n_tx,
        SUM(amount_usd) AS volume_usd,
        AVG(amount_usd) AS avg_amount,
        SUM(anti_fraud_status = 'positive') / COUNT(*) AS fraud_rate
    FROM `{table}`
    GROUP BY DATE(DATE_SUB(created, INTERVAL WEEKDAY(created) DAY))
    ORDER BY week_start
    """
)
weekly["week_start"] = pd.to_datetime(weekly["week_start"])

fig, ax = plt.subplots(figsize=(12, 4.4))
ax.plot(weekly["week_start"], weekly["n_tx"], marker="o", linewidth=2, color="#4C78A8")
ax.set_title("Transactions per week")
ax.set_xlabel("")
ax.set_ylabel("transactions")
ax.yaxis.set_major_formatter(lambda x, _: f"{x/1e3:.0f}k")
show(fig)

In [ ]:
daily = read_sql(
    f"""
    SELECT
        DATE(created) AS day,
        COUNT(*) AS n_tx,
        SUM(amount_usd) AS volume_usd
    FROM `{table}`
    GROUP BY DATE(created)
    ORDER BY day
    """
)
daily["day"] = pd.to_datetime(daily["day"])

fig, ax = plt.subplots(figsize=(12, 4.4))
ax.plot(daily["day"], daily["n_tx"], linewidth=1.2, color="#4C78A8")
ax.set_title("Daily transaction volume")
ax.set_xlabel("")
ax.set_ylabel("transactions")
ax.yaxis.set_major_formatter(lambda x, _: f"{x/1e3:.0f}k")
show(fig)

In [ ]:
dow = read_sql(
    f"""
    SELECT
        DAYOFWEEK(created) AS dow_num,
        DAYNAME(created) AS dow_name,
        COUNT(*) AS n_tx,
        AVG(amount_usd) AS avg_amount
    FROM `{table}`
    GROUP BY DAYOFWEEK(created), DAYNAME(created)
    ORDER BY dow_num
    """
)
hourly = read_sql(
    f"""
    SELECT HOUR(created) AS hour, COUNT(*) AS n_tx, AVG(amount_usd) AS avg_amount
    FROM `{table}`
    GROUP BY HOUR(created)
    ORDER BY hour
    """
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
sns.barplot(data=dow, x="dow_name", y="n_tx", ax=axes[0], color="#54A24B")
axes[0].set_title("Volume by day of week")
axes[0].set_xlabel("")
axes[0].set_ylabel("transactions")
axes[0].tick_params(axis="x", rotation=30)

sns.barplot(data=hourly, x="hour", y="n_tx", ax=axes[1], color="#B279A2")
axes[1].set_title("Volume by hour of day")
axes[1].set_xlabel("hour")
axes[1].set_ylabel("transactions")
show(fig)

## Corridors (country + currency)

In [ ]:
corridors = read_sql(
    f"""
    SELECT
        payout_country,
        payout_currency,
        CONCAT(payout_country, '-', payout_currency) AS corridor,
        COUNT(*) AS n_tx,
        COUNT(DISTINCT sender_id) AS n_senders,
        MIN(amount_usd) AS amount_min,
        MAX(amount_usd) AS amount_max,
        AVG(amount_usd) AS amount_avg,
        SUM(amount_usd) AS volume_usd,
        SUM(anti_fraud_status = 'positive') / COUNT(*) AS fraud_rate
    FROM `{table}`
    GROUP BY payout_country, payout_currency
    ORDER BY n_tx DESC
    """
)
corridors["share"] = corridors["n_tx"] / corridors["n_tx"].sum()
print(f"corridors: {len(corridors)}")
corridors.head(15)

In [ ]:
top_corridors = corridors.head(20)

fig, ax = plt.subplots(figsize=(11, 6.2))
sns.barplot(data=top_corridors, y="corridor", x="n_tx", ax=ax, color="#4C78A8")
ax.set_title("Top 20 corridors by transaction count")
ax.set_xlabel("transactions")
ax.set_ylabel("")
show(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.2))
sns.barplot(
    data=corridors.sort_values("amount_avg", ascending=False).head(20),
    y="corridor",
    x="amount_avg",
    ax=ax,
    color="#F58518",
)
ax.set_title("Average amount per corridor (top 20)")
ax.set_xlabel("average USD")
ax.set_ylabel("")
show(fig)

In [ ]:
countries = (
    corridors.groupby("payout_country", as_index=False)
    .agg(n_tx=("n_tx", "sum"), volume_usd=("volume_usd", "sum"), amount_avg=("amount_avg", "mean"))
    .sort_values("n_tx", ascending=False)
)
currencies = (
    corridors.groupby("payout_currency", as_index=False)
    .agg(n_tx=("n_tx", "sum"), volume_usd=("volume_usd", "sum"))
    .sort_values("n_tx", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.4))
sns.barplot(data=countries.head(15), y="payout_country", x="n_tx", ax=axes[0], color="#54A24B")
axes[0].set_title("Top countries")
axes[0].set_xlabel("transactions")
axes[0].set_ylabel("")

sns.barplot(data=currencies.head(15), y="payout_currency", x="n_tx", ax=axes[1], color="#B279A2")
axes[1].set_title("Top currencies")
axes[1].set_xlabel("transactions")
axes[1].set_ylabel("")
show(fig)

## Amounts and fees

In [ ]:
amount_stats = read_sql(
    f"""
    SELECT
        MIN(amount_usd) AS min_amount,
        AVG(amount_usd) AS avg_amount,
        MAX(amount_usd) AS max_amount,
        STDDEV_SAMP(amount_usd) AS std_amount,
        MIN(fee_usd) AS min_fee,
        AVG(fee_usd) AS avg_fee,
        MAX(fee_usd) AS max_fee
    FROM `{table}`
    """
)
amount_sample = read_sql(
    f"""
    SELECT amount_usd
    FROM `{table}`
    WHERE id % 50 = 0
    """
)
pct = amount_sample["amount_usd"].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
pct.index = [f"p{int(i*100):02d}_sample" for i in pct.index]
pd.concat(
    [amount_stats.T.rename(columns={0: "value"}), pct.rename("value").to_frame()],
    axis=0,
)

In [ ]:
amount_hist = read_sql(
    f"""
    SELECT
        CASE
            WHEN amount_usd < 25 THEN '00-25'
            WHEN amount_usd < 50 THEN '25-50'
            WHEN amount_usd < 100 THEN '50-100'
            WHEN amount_usd < 200 THEN '100-200'
            WHEN amount_usd < 500 THEN '200-500'
            WHEN amount_usd < 1000 THEN '500-1k'
            WHEN amount_usd < 2500 THEN '1k-2.5k'
            WHEN amount_usd < 5000 THEN '2.5k-5k'
            ELSE '5k+'
        END AS bucket,
        CASE
            WHEN amount_usd < 25 THEN 1
            WHEN amount_usd < 50 THEN 2
            WHEN amount_usd < 100 THEN 3
            WHEN amount_usd < 200 THEN 4
            WHEN amount_usd < 500 THEN 5
            WHEN amount_usd < 1000 THEN 6
            WHEN amount_usd < 2500 THEN 7
            WHEN amount_usd < 5000 THEN 8
            ELSE 9
        END AS bucket_ord,
        COUNT(*) AS n_tx
    FROM `{table}`
    GROUP BY bucket, bucket_ord
    ORDER BY bucket_ord
    """
)

fig, ax = plt.subplots()
sns.barplot(data=amount_hist, x="bucket", y="n_tx", ax=ax, color="#4C78A8")
ax.set_title("Amount distribution (USD buckets)")
ax.set_xlabel("amount USD")
ax.set_ylabel("transactions")
ax.yaxis.set_major_formatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}k")
show(fig)
amount_hist

In [ ]:
log_hist = read_sql(
    f"""
    SELECT
        ROUND(LOG10(GREATEST(amount_usd, 0.01)), 1) AS log10_amount,
        COUNT(*) AS n_tx
    FROM `{table}`
    GROUP BY ROUND(LOG10(GREATEST(amount_usd, 0.01)), 1)
    ORDER BY log10_amount
    """
)

fig, ax = plt.subplots()
ax.fill_between(log_hist["log10_amount"], log_hist["n_tx"], step="mid", alpha=0.35, color="#4C78A8")
ax.plot(log_hist["log10_amount"], log_hist["n_tx"], drawstyle="steps-mid", color="#4C78A8")
ax.set_title("Amount distribution on log10 scale")
ax.set_xlabel("log10(amount USD)")
ax.set_ylabel("transactions")
show(fig)

In [ ]:
fee_hist = read_sql(
    f"""
    SELECT
        CASE
            WHEN fee_usd < 2 THEN '0-2'
            WHEN fee_usd < 4 THEN '2-4'
            WHEN fee_usd < 6 THEN '4-6'
            WHEN fee_usd < 8 THEN '6-8'
            WHEN fee_usd < 12 THEN '8-12'
            WHEN fee_usd < 20 THEN '12-20'
            ELSE '20+'
        END AS bucket,
        CASE
            WHEN fee_usd < 2 THEN 1
            WHEN fee_usd < 4 THEN 2
            WHEN fee_usd < 6 THEN 3
            WHEN fee_usd < 8 THEN 4
            WHEN fee_usd < 12 THEN 5
            WHEN fee_usd < 20 THEN 6
            ELSE 7
        END AS bucket_ord,
        COUNT(*) AS n_tx
    FROM `{table}`
    GROUP BY bucket, bucket_ord
    ORDER BY bucket_ord
    """
)
fee_ratio = read_sql(
    f"""
    SELECT
        CASE
            WHEN fee_usd / NULLIF(amount_usd, 0) < 0.01 THEN '<1%'
            WHEN fee_usd / NULLIF(amount_usd, 0) < 0.02 THEN '1-2%'
            WHEN fee_usd / NULLIF(amount_usd, 0) < 0.04 THEN '2-4%'
            WHEN fee_usd / NULLIF(amount_usd, 0) < 0.06 THEN '4-6%'
            WHEN fee_usd / NULLIF(amount_usd, 0) < 0.10 THEN '6-10%'
            ELSE '10%+'
        END AS bucket,
        CASE
            WHEN fee_usd / NULLIF(amount_usd, 0) < 0.01 THEN 1
            WHEN fee_usd / NULLIF(amount_usd, 0) < 0.02 THEN 2
            WHEN fee_usd / NULLIF(amount_usd, 0) < 0.04 THEN 3
            WHEN fee_usd / NULLIF(amount_usd, 0) < 0.06 THEN 4
            WHEN fee_usd / NULLIF(amount_usd, 0) < 0.10 THEN 5
            ELSE 6
        END AS bucket_ord,
        COUNT(*) AS n_tx
    FROM `{table}`
    GROUP BY bucket, bucket_ord
    ORDER BY bucket_ord
    """
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
sns.barplot(data=fee_hist, x="bucket", y="n_tx", ax=axes[0], color="#F58518")
axes[0].set_title("Fee distribution (USD)")
axes[0].set_xlabel("fee USD")
axes[0].set_ylabel("transactions")

sns.barplot(data=fee_ratio, x="bucket", y="n_tx", ax=axes[1], color="#E45756")
axes[1].set_title("Fee / amount ratio")
axes[1].set_xlabel("fee ratio")
axes[1].set_ylabel("transactions")
show(fig)

## Per-user behaviour

In [ ]:
users = read_sql(
    f"""
    SELECT
        sender_id,
        COUNT(*) AS n_tx,
        AVG(amount_usd) AS avg_amount,
        SUM(amount_usd) AS volume_usd,
        MIN(created) AS first_seen,
        MAX(created) AS last_seen,
        SUM(anti_fraud_status = 'positive') AS n_fraud
    FROM `{table}`
    GROUP BY sender_id
    """
)
user_summary = users[["n_tx", "avg_amount", "volume_usd", "n_fraud"]].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)
print(
    f"users={len(users):,}  "
    f"avg tx/user={users['n_tx'].mean():.2f}  "
    f"median tx/user={users['n_tx'].median():.0f}  "
    f"avg amount/user={users['avg_amount'].mean():.2f}"
)
user_summary

In [ ]:
tx_per_user = (
    users.assign(
        bucket=pd.cut(
            users["n_tx"],
            bins=[0, 1, 2, 5, 10, 20, 50, 100, 10_000],
            labels=["1", "2", "3-5", "6-10", "11-20", "21-50", "51-100", "100+"],
            include_lowest=True,
        )
    )
    .groupby("bucket", observed=True)
    .size()
    .reset_index(name="n_users")
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
sns.barplot(data=tx_per_user, x="bucket", y="n_users", ax=axes[0], color="#4C78A8")
axes[0].set_title("Transactions per user")
axes[0].set_xlabel("tx count")
axes[0].set_ylabel("users")

sns.histplot(users["avg_amount"].clip(upper=users["avg_amount"].quantile(0.99)), bins=40, ax=axes[1], color="#F58518")
axes[1].set_title("Average amount per user (clipped at p99)")
axes[1].set_xlabel("average USD")
axes[1].set_ylabel("users")
show(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
sample = users.sample(n=min(25_000, len(users)), random_state=42)
ax.scatter(sample["n_tx"], sample["avg_amount"], s=8, alpha=0.25, color="#4C78A8")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("User activity vs average amount")
ax.set_xlabel("transactions per user")
ax.set_ylabel("average amount USD")
show(fig)

## Status and labels

In [ ]:
status = read_sql(
    f"""
    SELECT status, COUNT(*) AS n_tx, AVG(amount_usd) AS avg_amount
    FROM `{table}`
    GROUP BY status
    ORDER BY n_tx DESC
    """
)
labels = read_sql(
    f"""
    SELECT anti_fraud_status, compliance_status, COUNT(*) AS n_tx
    FROM `{table}`
    GROUP BY anti_fraud_status, compliance_status
    ORDER BY n_tx DESC
    """
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
sns.barplot(data=status, x="status", y="n_tx", ax=axes[0], color="#72B7B2")
axes[0].set_title("Payout status")
axes[0].set_xlabel("")
axes[0].set_ylabel("transactions")

label_pivot = labels.pivot(index="anti_fraud_status", columns="compliance_status", values="n_tx").fillna(0)
sns.heatmap(label_pivot, annot=True, fmt=",.0f", cmap="Blues", ax=axes[1])
axes[1].set_title("Anti-fraud vs compliance")
show(fig)
status

In [ ]:
labels

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4.6))
ax1.plot(weekly["week_start"], weekly["n_tx"], color="#4C78A8", marker="o", label="transactions")
ax1.set_ylabel("transactions", color="#4C78A8")
ax2 = ax1.twinx()
ax2.plot(weekly["week_start"], weekly["fraud_rate"] * 100, color="#E45756", marker="s", label="fraud rate")
ax2.set_ylabel("fraud rate %", color="#E45756")
ax1.set_title("Weekly volume vs anti-fraud positive rate")
ax1.set_xlabel("")
show(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.2))
top_fraud = corridors.sort_values("n_tx", ascending=False).head(20)
sns.barplot(data=top_fraud, y="corridor", x="fraud_rate", ax=ax, color="#E45756")
ax.set_title("Anti-fraud positive rate in top 20 corridors")
ax.set_xlabel("fraud rate")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(lambda x, _: f"{x:.1%}")
show(fig)